In [1]:
# --- Importação de Bibliotecas Padrão e de Terceiros ---
import os
import sys


# --- Configuração do Caminho do Projeto para Importações Locais ---
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

In [2]:
df_itens = pd.read_csv(r"C:\repositorio\public-data-analysis-anomaly-detection-ml\data\processed\itens_notas_fiscais.csv",sep=";",encoding="utf-8")
df_notas = pd.read_csv(r"C:\repositorio\public-data-analysis-anomaly-detection-ml\data\processed_isolation_forest\notas_fiscais_com_anomalias.csv",sep=";",encoding="utf-8")

In [3]:
df_itens.head(1)

,descricaoProdutoServico,CHAVE_NF,codigoNcmSh,ncmSh,cfop,quantidade,unidade,valorUnitario,valor
0,CANETA RETROPROJETOR,35211169207850000161550010000008501783475076,96082000.0,"Canetas e marcadores, com ponta de feltro ou c...",6949.0,6.0,UN,3.95,23.7


In [4]:
df_notas.head(1)

,ID,ORGAO,FORNECEDOR,CNPJ,MUNICIPIO,VALOR_NF,TIPO_EVENTO,DATA,CHAVE_NF,MUNICIPIO_MUN,...,COD_IBGE,ANO_MES,ANO,MES,DIA_SEMANA,DIA_MES,VALOR_NF_POR_POPULACAO,ANOMALY_PREDICTION,ANOMALY_SCORE,POSSIVEL_ANOMALIA
0,1950,Ministério da Saúde - Unidades com vínculo direto,RCA PRODUTOS E SERVICOS LTDA,69207850000161,SANTA BARBARA D'OESTE,16004.55,Autorização de Uso,2021-11-01,35211169207850000161550010000008501783475076,SANTA BARBARA D'OESTE,...,3545803,2021-11,2021,11,0,1,84.52899,1,0.024948,Não


In [5]:
# === 2. Filtrar notas suspeitas ===
df_suspeitas = df_notas[df_notas["POSSIVEL_ANOMALIA"].str.lower() == "sim"]

# === 3. Trazer os itens dessas notas ===
df_itens_suspeitos = df_itens.merge(
    df_suspeitas[["CHAVE_NF"]],
    on="CHAVE_NF",
    how="inner"
)

In [6]:
df_itens_suspeitos.head(1)

,descricaoProdutoServico,CHAVE_NF,codigoNcmSh,ncmSh,cfop,quantidade,unidade,valorUnitario,valor
0,ETOMIDATO 2 MG / ML Inj. LISTA C1 CRT c/5 Ampolas,35211058430828000160550010002022671964431820,30049069.0,Outros medicamentos contendo compostos heteroc...,5923.0,4.0,KG,60.0,240.0


In [ ]:
# === 4. Agora, analisar item a item por NCM ===
resultados = []

for ncm, grupo in df_itens.groupby("codigoNcmSh"):

    # Remover valores ausentes
    grupo = grupo.dropna(subset=["valorUnitario"])
    if grupo.shape[0] < 10:  # precisa ter um histórico mínimo
        continue

    # Normalizar valores
    X = grupo[["valorUnitario"]].values
    X_scaled = StandardScaler().fit_transform(X)

    # Treinar DBSCAN
    dbscan = DBSCAN(eps=0.8, min_samples=5)  # pode ajustar eps
    labels = dbscan.fit_predict(X_scaled)

    # Adicionar resultados
    grupo["cluster"] = labels
    grupo["is_outlier_dbscan"] = grupo["cluster"] == -1
    grupo["codigoNcmSh"] = ncm

    resultados.append(grupo)

In [ ]:
# === 5. Unir tudo ===
df_itens_com_clusters = pd.concat(resultados, ignore_index=True)

# === 6. Filtrar apenas os itens das notas suspeitas ===
df_itens_suspeitos_final = df_itens_com_clusters.merge(
    df_itens_suspeitos[["chave_nfe", "codigoNcmSh"]],
    on=["chave_nfe", "codigoNcmSh"],
    how="inner"
)

In [ ]:
# === 7. Ver quais são outliers dentro das notas suspeitas ===
df_anomalias_confirmadas = df_itens_suspeitos_final[df_itens_suspeitos_final["is_outlier_dbscan"]]

print(df_anomalias_confirmadas[["chave_nfe", "codigoNcmSh", "valorUnitario", "is_outlier_dbscan"]].head())

In [ ]:
#Amostra aleatória de 5 anomalias confirmadas
df_anomalias_confirmadas.sample(5)

In [ ]:
# Definir diretório de saída (pasta processed dentro de data)
output_dir = os.path.join(project_root, "data", "processed_DBSCAN_final")

# Salvar o dataset com as anomalias detectadas para a próxima fase
df_anomalias_confirmadas.to_csv(os.path.join(output_dir,"df_anomalias_confirmadas.csv"), index=False, sep=";", encoding="utf-8")
print("Dataset com anomalias salvo como df_anomalias_confirmadas.csv")